In [1]:
import requests
import os
import re
import json
from pypdf import PdfReader
import uuid
import base64
from dotenv import load_dotenv,find_dotenv
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Dict, Any, List
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer  
from langgraph.checkpoint.sqlite import SqliteSaver 

e:\resume_talk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(find_dotenv(),override=True)

RAPID_API_KEY = os.getenv('RAPID_API_KEY')
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

print(f"Rapid Key Found: {RAPID_API_KEY is not None}")
print(f"Gemini Key Found: {GOOGLE_API_KEY is not None}")
print(f"GROQ Key Found: {GROQ_API_KEY is not None}")

Rapid Key Found: True
Gemini Key Found: True
GROQ Key Found: True


In [3]:


# Set GROQ_API_KEY in environment or pass directly
llm_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
)
llm_google=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.0
)

# rag 

In [ ]:


def extract_profile(text: str):
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    name = lines[0] if lines else None

    email = None
    phone = None

    for line in lines:
        if not email:
            m = re.search(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", line)
            if m:
                email = m.group()

        if not phone:
            m = re.search(r"\+?\d[\d\s\-]{8,}\d", line)
            if m:
                phone = m.group()

    return {
        "name": name,
        "email": email,
        "phone": phone,
    }


In [ ]:


SECTIONS = {
    "PROJECTS",
    "SKILLS",
    "EDUCATION",
    "EXPERIENCE",
    "CERTIFICATIONS",
    "INTERNSHIPS",
}

def sectionize_documents(docs):
    section_docs = []

    for doc in docs:
        current_section = "GENERAL"
        buffer = []

        for line in doc.page_content.splitlines():
            clean = line.strip()
            if not clean:
                continue

            if clean.upper() in SECTIONS:
                if buffer:
                    section_docs.append(
                        Document(
                            page_content="\n".join(buffer),
                            metadata={**doc.metadata, "section": current_section}
                        )
                    )
                current_section = clean.upper()
                buffer = []
            else:
                buffer.append(clean)

        if buffer:
            section_docs.append(
                Document(
                    page_content="\n".join(buffer),
                    metadata={**doc.metadata, "section": current_section}
                )
            )

    return section_docs


In [ ]:
def load_documents(path:Path):
    loader=PyPDFLoader(
        str(path),
    )

    return loader.load()

In [ ]:
def split_documents(documents):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", " ", ""],
    )

    return splitter.split_documents(documents)

In [ ]:
def create_vector_store(docs):
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return FAISS.from_documents(docs, embeddings)

def get_retriever(vs):
    return vs.as_retriever(search_kwargs={"k": 5})

In [ ]:
def route_query(q: str):
    q = q.lower()
    if "name" in q:
        return "NAME"
    if "email" in q:
        return "EMAIL"
    if "phone" in q:
        return "PHONE"
    if "project" in q:
        return "PROJECTS"
    if "skill" in q:
        return "SKILLS"
    if "education" in q:
        return "EDUCATION"
    return "RAG"

In [ ]:
class RAGSERVICE:

    def __init__(self, data_path: Path, llm_google):
        self.data_path = data_path
        self.llm_google = llm_google
        self.retriever = None
        self.profile = None

    def _build(self):
        docs = load_documents(self.data_path)
        if not docs:
            raise RuntimeError("No documents loaded")

        full_text = "\n".join(d.page_content for d in docs)
        self.profile = extract_profile(full_text)

        section_docs = sectionize_documents(docs)
        chunks = split_documents(section_docs)
        vs = create_vector_store(chunks)
        self.retriever = get_retriever(vs)

    def query(self, question: str):
        intent = route_query(question)

        if intent == "NAME":
            return self.profile["name"]

        if intent == "EMAIL":
            return self.profile["email"]

        if intent == "PHONE":
            return self.profile["phone"]

        docs = self.retriever.invoke(question)
        context = "\n".join(d.page_content for d in docs)

        prompt = f"""
You are given resume content.

Context:
{context}

Question:
{question}

Answer concisely using ONLY the context.
"""

        return self.llm_google.invoke(prompt)

    def rebuild(self):
        self._build()


In [ ]:
BASE_DIR = Path.cwd().parents[0] 
file_path= BASE_DIR / "data" / "Utkarsh_Singh_CV.pdf"

if not file_path.exists():
    raise FileNotFoundError(f"{file_path} not found")

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
rag=RAGSERVICE(file_path,llm)
rag._build()

In [ ]:
print(rag.query('Projects?'))

# ollama

In [4]:
class ResumeState(TypedDict):
    raw_pdf_path: str
    raw_text: str
    sections: Dict[str, str]
    entities: Dict[str, Any]
    signals: Dict[str, Any]
    evaluation: Dict[str, Any]
    ats_score: int
    ats_breakdown: Dict[str, Any]
    final_output: str
    job_role: str
    job_level: str
    current_date: str
    iteration: int
    recommendations: List[str]
    job_openings: List[dict]
    email_draft: str
    next_action: str
    jobs_fetched: bool          # new
    emails_drafted: int

In [5]:

def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages).strip()

def ingest_resume(state: ResumeState):
    text = extract_text_from_pdf(state["raw_pdf_path"])
    return {
        "raw_text": text,
        "iteration": 0,
        "current_date": "2026-03-01",  # hardcode or use datetime.now()
        "recommendations": [],
        "job_openings": [],
        "email_draft": ""
    }

In [6]:
def detect_sections(state: ResumeState):
    prompt = f"""
Split the resume into sections: header, skills, experience, projects, education, certifications.
Resume:
{state["raw_text"]}

Return JSON only.
"""
    resp = llm_google.invoke(prompt)
    try:
        sections = json.loads(resp.content)
    except:
        sections = {}
    return {"sections": sections}

In [7]:

def extract_entities(state: ResumeState):
    prompt = f"""
Extract entities: languages, frameworks, tools, databases, domains.
Sections:
{state["sections"]}

Return JSON only.
"""
    resp = llm_google.invoke(prompt)
    try:
        entities = json.loads(resp.content)
    except:
        entities = {}
    return {"entities": entities}

In [8]:
## doubtfull.

def infer_signals(state: ResumeState):
    prompt = f"""
Infer signals: seniority_level, role_bias, project_quality, impact_strength.
Entities: {state["entities"]}
Sections: {state["sections"]}

Return JSON only.
"""
    resp = llm_google.invoke(prompt)
    try:
        signals = json.loads(resp.content)
    except:
        signals = {}
    return {"signals": signals}

In [9]:
def evaluate_resume(state: ResumeState):
    prompt = f"""
Critically evaluate.
Signals: {state["signals"]}
Return JSON: strengths, weaknesses, missing_elements, improvement_advice
"""
    resp = llm_google.invoke(prompt)
    try:
        eval_data = json.loads(resp.content)
    except:
        eval_data = {}
    return {"evaluation": eval_data}

In [10]:
def ats_scoring(state: ResumeState):
    prompt = f"""
    You are a strict ATS scoring engine.

    Job role: {state["job_role"]}
    Level: {state["job_level"]}
    Current date: {state["current_date"]}

    Resume text:
    {state["raw_text"]}

    Score 0-100. Be harsh.
    Return ONLY JSON:
    {{
      "score": <int 0-100>,
      "breakdown": {{"skills": <0-40>, "domain": <0-20>, "role": <0-20>, "completeness": <0-10>}},
      "missing_keywords": ["..."],
      "penalty_reason": "short reason or empty"
    }}
    """

    resp = llm_groq.invoke(prompt)
    raw = resp.content.strip()

    print("[ATS] Raw LLM output:", repr(raw[:600]))   # <--- keep this for now

    # More aggressive cleaning
    cleaned = raw
    # Remove common markdown fences
    # Remove any leading/trailing backticks or code words
    cleaned = re.sub(r'^(```|json|\s*)+', '', cleaned)
    cleaned = re.sub(r'(```|json|\s*)+$', '', cleaned)
    cleaned = cleaned.strip()

    try:
        data = json.loads(cleaned)
        score = int(data.get("score", 0))
        if score < 0 or score > 100:
            score = 50  # clamp invalid values
        return {
            "ats_score": score,
            "ats_breakdown": data
        }
    except json.JSONDecodeError as e:
        print("[ATS PARSE ERROR]", str(e))
        print("Failed cleaned text:", repr(cleaned[:400]))
        # Never return 0 — it's too punishing
        return {
            "ats_score": 50,   # neutral fallback
            "ats_breakdown": {
                "error": "JSON parse failed",
                "raw": raw[:300],
                "cleaned_snippet": cleaned[:200]
            }
        }
    except Exception as e:
        print("[ATS UNEXPECTED ERROR]", type(e).__name__, str(e))
        return {"ats_score": 40, "ats_breakdown": {"unexpected_error": str(e)}}

In [13]:
def supervisor(state: ResumeState):
    # Extract values with defaults to prevent KeyErrors
    score = state.get("ats_score", 0)
    iteration = state.get("iteration", 0)
    jobs_fetched = state.get("jobs_fetched", False)
    emails_drafted = state.get("emails_drafted", 0)
    
    # 1. Force Stop Condition
    if iteration >= 3:
        return {"next_action": "FINAL"}
    
    # 2. Success / "Good Enough" Conditions
    # Score >= 80 OR (Score >= 70 after 2 attempts)
    is_passing_score = score >= 80 or (score >= 70 and iteration >= 2)
    
    if is_passing_score:
        if not jobs_fetched:
            return {"next_action": "JOBS"}
        elif emails_drafted >= 1:
            return {"next_action": "FINAL"}
        else:
            return {"next_action": "EMAIL"}

    # 3. Default fallback: Refine the resume
    return {"next_action": "REFINE"}

In [14]:
def refine_resume(state: ResumeState):
    iteration = state["iteration"] + 1
    max_iter = 3
    if iteration > max_iter:
        print(f"Max iterations ({max_iter}) reached → forcing FINAL")
        return {
            "iteration": iteration,
            "next_action": "FINAL"   
        }

    prompt = f"""
Improve resume for {state["job_role"]} {state["job_level"]}.
Score: {state["ats_score"]}
Breakdown: {state["ats_breakdown"]}

Rewrite raw text to fix gaps.
Return only the new raw_text string.
"""
    new_text = llm_groq.invoke(prompt).content.strip()
    return {
        "raw_text": new_text,
        "iteration": iteration,
        "recommendations": state["recommendations"] + [f"Refined iteration {iteration}"]
    }

In [15]:
def fetch_job_openings(state: ResumeState):
    url = "https://jsearch.p.rapidapi.com/search"
    query = f"{state['job_role']} {state['job_level']} India"
    params = {"query": query, "page": "1", "num_pages": "3"}
    headers = {
        "X-RapidAPI-Key": os.getenv("RAPID_API_KEY"),
        "X-RapidAPI-Host": "jsearch.p.rapidapi.com"
    }
    try:
        r = requests.get(url, headers=headers, params=params).json()
        jobs = r.get("data", [])[:3]
        formatted = [{"title": j["job_title"], "company": j["employer_name"], "link": j["job_apply_link"]} for j in jobs]
        return {"job_openings": formatted,"jobs_fetched": True}
    except:
        return {"job_openings": []}

In [16]:

def draft_email(state: ResumeState):
    if state["ats_score"] < 80 or not state["job_openings"]:
        return {"email_draft": "Score too low or no jobs — no draft"}
    
    job = state["job_openings"][0]
    prompt = f"""
Draft short email for {state["job_role"]} at {job["company"]}.

Subject, greeting, 2-3 sentences why fit, attach resume, closing.
"""
    # resp = llm_google.invoke(prompt)
    resp = llm_groq.invoke(prompt)
    return {"email_draft": resp.content.strip()}

In [17]:
def generate_final_output(state: ResumeState):
    prompt = f"""
Summarize resume optimization.

Score: {state["ats_score"]}
Breakdown: {state["ats_breakdown"]}
Recommendations: {state["recommendations"]}
Jobs: {state["job_openings"]}
Email draft: {state.get("email_draft", "none")}

Return concise markdown report.
"""
    resp = llm_google.invoke(prompt)
    return {"final_output": resp.content.strip()}

In [19]:

# Graph
graph = StateGraph(ResumeState)

graph.add_node("ingest", ingest_resume)
graph.add_node("sections", detect_sections)
graph.add_node("entities", extract_entities)
graph.add_node("signals", infer_signals)
graph.add_node("evaluate", evaluate_resume)
graph.add_node("ats", ats_scoring)
graph.add_node("supervisor", supervisor)
graph.add_node("refine_resume", refine_resume)
graph.add_node("fetch_job_openings", fetch_job_openings)
graph.add_node("draft_email", draft_email)
graph.add_node("final", generate_final_output)

graph.set_entry_point("ingest")

graph.add_edge("ingest", "sections")
graph.add_edge("sections", "entities")
graph.add_edge("entities", "signals")
graph.add_edge("signals", "evaluate")
graph.add_edge("evaluate", "ats")
graph.add_edge("ats", "supervisor")

graph.add_conditional_edges(
    "supervisor",
    lambda s: s["next_action"],
    {
        "REFINE": "refine_resume",
        "JOBS": "fetch_job_openings",
        "EMAIL": "draft_email",
        "FINAL": "final"
    }
)

graph.add_edge("refine_resume", "ats")
graph.add_edge("fetch_job_openings", "draft_email")
graph.add_edge("draft_email", "final")
graph.add_edge("final", END)


In [20]:
def get_id():
    id=uuid.uuid4()
    return id

In [21]:
with SqliteSaver.from_conn_string('resume_databse.db') as checkpointer:
    app=graph.compile(checkpointer=checkpointer)

    thread_id=get_id()
    config={"configurable": {"thread_id": thread_id}}

    initial_input = {
    "raw_pdf_path": "Utkarsh_Singh_CV.pdf",
    "job_role": "Machine Learning and Ai Engineer",
    "job_level": "Fresher",
    }

    for event in app.stream(
        initial_input,
        config,
        stream_mode="updates"          
    ):
        for node_name, update in event.items():
            print(f"→ Node finished: {node_name}")
            if update:
                print("  Updated keys:", list(update.keys()))
                if "ats_score" in update:
                    print(f"  ATS score became: {update['ats_score']}")
                if "next_action" in update:
                    print(f"  Supervisor decided: {update.get('next_action')}")
        print("-" * 60)


→ Node finished: ingest
  Updated keys: ['raw_text', 'iteration', 'current_date', 'recommendations', 'job_openings', 'email_draft']
------------------------------------------------------------
→ Node finished: sections
  Updated keys: ['sections']
------------------------------------------------------------
→ Node finished: entities
  Updated keys: ['entities']
------------------------------------------------------------
→ Node finished: signals
  Updated keys: ['signals']
------------------------------------------------------------
→ Node finished: evaluate
  Updated keys: ['evaluation']
------------------------------------------------------------
[ATS] Raw LLM output: '```\n{\n  "score": 82,\n  "breakdown": {\n    "skills": 36,\n    "domain": 18,\n    "role": 18,\n    "completeness": 10\n  },\n  "missing_keywords": ["computer vision", "reinforcement learning", "natural language processing"],\n  "penalty_reason": "lacking diversity in projects and limited experience"\n}\n```'
→ Node f